# 01 — Explore entities, descriptions, sequences, and embeddings

Entity tables bind stable identifiers to cross-references and labels. Feature tables carry modality-specific data such as text, sequence, fingerprints, or embeddings. Modalities stay separate because sequence similarity, textual similarity, and chemical similarity answer different questions.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "public":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


In [ ]:
root = str(KG_ROOT).rstrip("/")
gene_uri = f"{root}/nodes/gene.parquet"
genes = read_bounded_parquet(gene_uri, limit=10, billing_project=BILLING_PROJECT)
display(genes[[column for column in ["id", "gene_name", "name", "description", "source"] if column in genes]])


In [ ]:
feature_uri = f"{root}/features/gene_textual_summary.parquet"
features = read_bounded_parquet(feature_uri, limit=10, billing_project=BILLING_PROJECT)
display(features)
print("Canonical protein/transcript sequence tables can be sampled the same way; never call an unbounded full-table read from a laptop.")


In [ ]:
from manage_db.public_notebooks import nearest_embeddings

if MODE == "fixture":
    embedding_uri = Path(KG_ROOT) / "features" / "embeddings" / "text" / "fixture.parquet"
else:
    embedding_uri = os.environ.get("JOUVENCE_EMBEDDING_URI")
    if not embedding_uri:
        raise RuntimeError("Set JOUVENCE_EMBEDDING_URI to an accepted immutable public embedding shard; publication is not assumed.")
neighbors = nearest_embeddings(embedding_uri, "ENSG00000012048", limit=5, billing_project=BILLING_PROJECT)
display(neighbors)


Cosine neighbors are candidates for inspection, not functional equivalence. Similarity depends on encoder, source text/sequence, preprocessing, and coverage. A learned fallback permits model execution but must not be described as source-derived biology.
